Welcome to this notebook, where we conduct experiments on tabular datasets.

In [3]:
import secrets
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_wine, load_breast_cancer, load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

import nll_to_po.training.loss as L
import nll_to_po.training.reward as R
import nll_to_po.models.dn_policy as Policy
from nll_to_po.training.utils import train_single_policy, set_seed_everywhere

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = secrets.randbits(32)
set_seed_everywhere(seed)
print(f"Seed: {seed}")

sns.set_theme(style="ticks", font_scale=1.2)
sns.set_palette("colorblind")

Seed: 441611831


In [4]:
UCI_NAME_TO_ID = {"credit_default": 350, "spambase": 94, "poker": 158}


def _ensure_numpy_X_y(X, y):
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy()
    if isinstance(y, (pd.Series, pd.DataFrame)):
        y = y.to_numpy().ravel()

    if y.dtype.kind not in "iu":
        le = LabelEncoder()
        y = le.fit_transform(y.astype(str))
    return X, y.astype(np.int64, copy=False)


def load_uci(
    dataset="wine",
    test_size=0.2,
    val_size=0.2,
    batch_size=256,
    standardize=True,
    impute_missing=False,
    impute_strategy="median",
    random_state=0,
    uci_id=None,
):
    if dataset in {"wine", "iris", "breast_cancer", "load_digits"} and uci_id is None:
        if dataset == "wine":
            data = load_wine()
        elif dataset == "iris":
            data = load_iris()
        elif dataset == "breast_cancer":
            data = load_breast_cancer()
        elif dataset == "load_digits":
            data = load_digits()
        X, y = data.data, data.target

    else:
        if uci_id is None:
            if dataset in UCI_NAME_TO_ID:
                uci_id = UCI_NAME_TO_ID[dataset]
            else:
                raise ValueError(
                    f"Unknown dataset '{dataset}'. "
                    f"Use one of {{'wine','iris','breast_cancer','load_digits'}} "
                    f"or provide a valid UCI id via `uci_id`."
                )
        ds = fetch_ucirepo(id=uci_id)
        X = ds.data.features
        y = ds.data.targets
        if isinstance(y, pd.DataFrame) and y.shape[1] > 1:
            y = y.iloc[:, 0]

        X, y = _ensure_numpy_X_y(X, y)

    if impute_missing:
        imp = SimpleImputer(strategy=impute_strategy)
        X = imp.fit_transform(X)

    if standardize:
        scaler = StandardScaler().fit(X)
        X = scaler.transform(X)

    X_tr, X_tt, y_tr, y_tt = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr, y_tr, test_size=val_size, stratify=y_tr, random_state=random_state
    )

    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    X_tt = torch.tensor(X_tt, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)
    y_tt = torch.tensor(y_tt, dtype=torch.long)

    tr_loader = DataLoader(
        TensorDataset(X_tr, y_tr, y_tr, torch.zeros_like(y_tr)),
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(X_val, y_val, y_val, torch.zeros_like(y_val)),
        batch_size=batch_size,
        shuffle=False,
    )
    tst_loader = DataLoader(
        TensorDataset(X_tt, y_tt, y_tt, torch.zeros_like(y_tt)),
        batch_size=batch_size,
        shuffle=False,
    )

    meta = {"input_dim": X.shape[1], "num_classes": int(np.unique(y).size)}
    return tr_loader, val_loader, tst_loader, meta

In [5]:
@torch.no_grad()
def _trace_from_counts(counts: torch.Tensor) -> float:
    N = int(counts.sum().item())
    if N <= 1:
        return 0.0
    p2_sum = (counts.float() / N).pow(2).sum().item()
    return float((N / (N - 1)) * (1.0 - p2_sum))


@torch.no_grad()
def estimate_trace_sigma_onehot(train_loader, num_classes: int) -> float:
    counts = torch.zeros(num_classes, dtype=torch.long)
    for batch in train_loader:
        yb = batch[1]
        if yb.ndim == 2 and yb.size(1) == num_classes:
            yb = yb.argmax(dim=1)
        yb = yb.to(torch.long).flatten()
        counts += torch.bincount(yb, minlength=num_classes)
    return _trace_from_counts(counts)


def beta_star_from_data(train_loader, entropy_weight: float, num_classes: int) -> float:
    tr = estimate_trace_sigma_onehot(train_loader, num_classes=num_classes)
    return float(entropy_weight * num_classes / (2.0 * max(tr, 1e-12)))

In [6]:
@torch.no_grad
def evaluate_accuracy_multi_regression(
    policy: nn.Module, data_loader: DataLoader
) -> float:
    policy.eval()
    correct, total = 0, 0
    for xb, yb, *rest in data_loader:
        _, prob_predit = policy(xb)
        y_prob = prob_predit.argmax(dim=-1)
        correct += (y_prob == yb).sum().item()
        total += yb.numel()
    if total == 1:
        return "One element in the dataloader"
    else:
        return correct / max(total, 1)

In [7]:
@torch.no_grad()
def _collect_binary_scores_and_labels(
    policy: nn.Module, data_loader: DataLoader, device: str = "cpu"
):
    policy.eval()
    ys, ss = [], []
    for xb, yb, *rest in data_loader:
        xb = xb.to(device)
        yb = yb.to(device).long().view(-1)
        _, out = policy(xb)

        if out.ndim == 1 or (out.ndim == 2 and out.size(1) == 1):
            out = out.view(-1)
            if (out.max() > 1) or (out.min() < 0):  # logits
                pos_prob = out.sigmoid()
            else:
                pos_prob = out.clamp(1e-12, 1 - 1e-12)
        elif out.ndim == 2 and out.size(1) == 2:
            probs = out.softmax(dim=1) if (out.min() < 0 or out.max() > 1) else out
            pos_prob = probs[:, 1].clamp(1e-12, 1 - 1e-12)
        else:
            raise ValueError(f"Unexpected output shape {out.shape}")

        ys.append(yb.cpu())
        ss.append(pos_prob.cpu())

    y_true = torch.cat(ys, 0).numpy()
    scores = torch.cat(ss, 0).numpy()
    return y_true, scores


def evaluate_tpr_tnr_binary(policy, data_loader, threshold=0.5, device="cpu"):
    y_true, scores = _collect_binary_scores_and_labels(policy, data_loader, device)
    y_pred = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    TPR = tp / (tp + fn) if (tp + fn) else 0.0
    TNR = tn / (tn + fp) if (tn + fp) else 0.0
    FPR = 1 - TNR
    FNR = 1 - TPR
    ACC = (tp + tn) / max(tp + tn + fp + fn, 1)
    BAL_ACC = (TPR + TNR) / 2.0

    return dict(
        TP=tp,
        TN=tn,
        FP=fp,
        FN=fn,
        TPR=TPR,
        TNR=TNR,
        FPR=FPR,
        FNR=FNR,
        ACC=ACC,
        BAL_ACC=BAL_ACC,
    )


def evaluate_roc_auc_binary(policy, data_loader, device="cpu"):
    y_true, scores = _collect_binary_scores_and_labels(policy, data_loader, device)
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)
    return dict(fpr=fpr, tpr=tpr, thresholds=thresholds, auc=auc)

Run the full training–evaluation pipeline on a single dataset, comparing **NLL** vs **PO_Entropy** (with `β ∈ {1, β⋆}`), and return learning curves, test metrics, and the estimated `β⋆`.

## Summary

- Loads the dataset and builds train/val/test loaders.
- Estimates the optimal entropy weight scaling `β⋆` from training data.
- Trains a baseline **NLL** classifier and two **PO_Entropy** classifiers (for `β=1` and `β=β⋆`) over `n_experiments` repetitions.
- Logs per-epoch train/val curves and per-rep test metrics (accuracy + binary metrics if `C=2`).


In [8]:
def run_one_dataset(
    dataset_name: str,
    n_updates: int = 10,
    n_experiments: int = 5,
    batch_size: int = 128,
    learning_rate: float = 1e-4,
    entropy_weight: float = 1e-3,
    type_of_data: str = "tabular",
):
    device = torch.device("cpu")

    if type_of_data == "tabular":
        print(f"Running on tabular dataset: {dataset_name}")
        train_loader, val_loader, test_loader, meta = load_uci(
            dataset=dataset_name,
            batch_size=batch_size,
            standardize=True,
            impute_missing=True,
        )
        C = meta["num_classes"]
        D_in = meta["input_dim"]

    print("finished to load datasets")
    beta_star = beta_star_from_data(
        train_loader, entropy_weight=entropy_weight, num_classes=C
    )
    print(f"Estimated beta_star: {beta_star:.4e}")
    beta_list = [1, beta_star]

    curves = []
    tests = []

    for rep in range(n_experiments):
        if type_of_data == "tabular":
            policy = Policy.MulticlassLogisticRegression(D_in, C)
        loss_fn = L.NLL_Classification()
        print("start training NLL")

        trained_policy, train_metrics, val_metrics, _, _ = train_single_policy(
            policy=policy,
            train_dataloader=train_loader,
            val_dataloader=val_loader,
            loss_function=loss_fn,
            n_updates=n_updates,
            learning_rate=learning_rate,
            wandb_run=None,
            tensorboard_writer=None,
            logger=None,
            scheduler_patience=20,
            early_stopping_patience=n_updates,
            device=device,
        )

        df_tr = (
            pd.DataFrame(train_metrics).reset_index().rename(columns={"index": "epoch"})
        )
        df_tr["split"] = "train"
        df_tr["method"] = "NLL"
        df_tr["beta"] = np.nan
        df_tr["rep"] = rep
        df_val = (
            pd.DataFrame(val_metrics).reset_index().rename(columns={"index": "epoch"})
        )
        df_val["split"] = "val"
        df_val["method"] = "NLL"
        df_val["beta"] = np.nan
        df_val["rep"] = rep
        curves += [df_tr, df_val]

        test_acc = evaluate_accuracy_multi_regression(trained_policy, test_loader)

        if C == 2:
            thr_metrics = evaluate_tpr_tnr_binary(
                trained_policy, test_loader, threshold=0.5, device=str(device)
            )
            roc_res = evaluate_roc_auc_binary(
                trained_policy, test_loader, device=str(device)
            )
            test_tpr = thr_metrics["TPR"]
            test_fnr = thr_metrics["FNR"]
            test_tnr = thr_metrics["TNR"]
            test_balacc = thr_metrics["BAL_ACC"]
            test_auc = roc_res["auc"]

        tests.append(
            {
                "dataset": dataset_name,
                "method": "NLL",
                "beta": np.nan,
                "rep": rep,
                "test_accuracy": test_acc,
                "test_tpr": test_tpr,
                "test_fnr": test_fnr,
                "test_tnr": test_tnr,
                "test_bal_acc": test_balacc,
                "test_auc": test_auc,
            }
        )

    for beta in beta_list:
        U = torch.eye(C) * float(beta)
        reward = R.OneHotMahalanobis(U, num_classes=C)

        for rep in range(n_experiments):
            if type_of_data == "tabular":
                policy = Policy.MulticlassLogisticRegression(D_in, C)

            loss_fn = L.PO_Entropy_Classification(
                reward_fn=reward,
                n_generations=50,
                use_rsample=False,
                reward_transform="none",
                entropy_weight=entropy_weight,
            )

            beta_trained_policy, train_metrics, val_metrics, _, _ = train_single_policy(
                policy=policy,
                train_dataloader=train_loader,
                val_dataloader=val_loader,
                loss_function=loss_fn,
                n_updates=n_updates,
                learning_rate=learning_rate,
                wandb_run=None,
                tensorboard_writer=None,
                logger=None,
                early_stopping_patience=n_updates,
                device=device,
            )

            df_tr = (
                pd.DataFrame(train_metrics)
                .reset_index()
                .rename(columns={"index": "epoch"})
            )
            df_tr["split"] = "train"
            df_tr["method"] = "PO_Entropy"
            df_tr["beta"] = beta
            df_tr["rep"] = rep
            df_val = (
                pd.DataFrame(val_metrics)
                .reset_index()
                .rename(columns={"index": "epoch"})
            )
            df_val["split"] = "val"
            df_val["method"] = "PO_Entropy"
            df_val["beta"] = beta
            df_val["rep"] = rep
            curves += [df_tr, df_val]

            test_acc = evaluate_accuracy_multi_regression(
                beta_trained_policy, test_loader
            )

            if C == 2:
                thr_metrics = evaluate_tpr_tnr_binary(
                    beta_trained_policy, test_loader, threshold=0.5, device=str(device)
                )
                roc_res = evaluate_roc_auc_binary(
                    beta_trained_policy, test_loader, device=str(device)
                )
                test_tpr = thr_metrics["TPR"]
                test_tnr = thr_metrics["TNR"]
                test_fnr = thr_metrics["FNR"]
                test_balacc = thr_metrics["BAL_ACC"]
                test_auc = roc_res["auc"]

            tests.append(
                {
                    "dataset": dataset_name,
                    "method": "PO_Entropy",
                    "beta": beta,
                    "rep": rep,
                    "test_accuracy": test_acc,
                    "test_tpr": test_tpr,
                    "test_tnr": test_tnr,
                    "test_fnr": test_fnr,
                    "test_bal_acc": test_balacc,
                    "test_auc": test_auc,
                }
            )

    curves_df = pd.concat(curves, ignore_index=True)
    tests_df = pd.DataFrame(tests)

    curves_df["is_beta_star"] = curves_df["beta"].apply(
        lambda b: isinstance(b, float) and abs(b - beta_star) < 1e-12
    )
    tests_df["is_beta_star"] = tests_df["beta"].apply(
        lambda b: isinstance(b, float) and abs(b - beta_star) < 1e-12
    )
    return curves_df, tests_df, beta_star

In [9]:
def plot_curves_for_dataset(
    curves_df: pd.DataFrame, dataset_name: str, beta_star: float
):
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

    def make_label(row):
        if row["method"] == "NLL":
            return "NLL"
        if pd.isna(row["beta"]):
            return "PO (β=NA)"
        if abs(row["beta"] - beta_star) < 1e-12:
            return "PO (β*)"
        return f"PO (β={row['beta']:.3g})"

    curves_df = curves_df.copy()
    curves_df["label"] = curves_df.apply(make_label, axis=1)

    palette_map = {}
    for lab in curves_df["label"].unique():
        if lab == "NLL":
            palette_map[lab] = "#1f77b4"
        elif lab == "PO (β*)":
            palette_map[lab] = "black"
        else:
            palette_map[lab] = "red"

    sub = curves_df[curves_df["split"] == "train"]
    sns.lineplot(
        data=sub,
        x="epoch",
        y="accuracy",
        hue="label",
        errorbar=("ci", 95),
        ax=ax[0],
        palette=palette_map,
        legend=False,
    )
    ax[0].set_title(f"{dataset_name}: Train accuracy vs epoch")
    ax[0].set_xlabel("epoch")
    ax[0].set_ylabel("accuracy")

    # VAL
    sub = curves_df[curves_df["split"] == "val"]
    sns.lineplot(
        data=sub,
        x="epoch",
        y="accuracy",
        hue="label",
        errorbar=("ci", 95),
        ax=ax[1],
        palette=palette_map,
        legend=True,
    )
    ax[1].set_title(f"{dataset_name}: Val accuracy vs epoch")
    ax[1].set_xlabel("epoch")
    ax[1].set_ylabel("accuracy")
    ax[1].legend(title="method", frameon=False, loc="lower right")

    plt.tight_layout()
    plt.show()

# Hyperparameters

| Dataset          | n_updates | n_experiments | batch_size | learning_rate | entropy_weight | type_of_data |
|------------------|-----------|---------------|------------|---------------|----------------|--------------|
| poker            | 30        | 3             | 128        | 4×10⁻²        | 21             | tabular      |
| credit_default   | 30        | 3             | 128        | 5×10⁻²        | 10             | tabular      |


In [15]:
datasets = ["credit_default"]

all_curves = []
all_tests = []
for ds in datasets:
    curves_df, tests_df, bstar = run_one_dataset(
        dataset_name=ds,
        n_updates=30,
        n_experiments=3,
        batch_size=128,
        learning_rate=4 * 1e-2,
        entropy_weight=10,
        type_of_data="tabular",
    )
    # plot_curves_for_dataset(curves_df, ds, bstar)
    curves_df["dataset"] = ds
    tests_df["dataset"] = ds
    all_curves.append(curves_df)
    all_tests.append(tests_df)

df_curves_all = pd.concat(all_curves, ignore_index=True)
df_tests_all = pd.concat(all_tests, ignore_index=True)

Running on tabular dataset: credit_default
finished to load datasets
Estimated beta_star: 2.9023e+01
start training NLL


Training epochs: 100%|██████████| 30/30 [00:09<00:00,  3.22it/s]


start training NLL


Training epochs: 100%|██████████| 30/30 [00:08<00:00,  3.38it/s]


start training NLL


Training epochs: 100%|██████████| 30/30 [00:15<00:00,  2.00it/s]


In [16]:
agg = (
    df_tests_all.groupby(["dataset", "method", "beta"], dropna=False)["test_accuracy"]
    .agg(["mean", "std"])
    .reset_index()
)

agg["beta"] = agg["beta"].apply(lambda b: "–" if pd.isna(b) else f"{b:.3f}")

# acc ± std
agg["acc ± std"] = agg.apply(
    lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
)

df_pretty = agg[["dataset", "method", "beta", "acc ± std"]]

In [17]:
df_pretty

,dataset,method,beta,acc ± std
0,credit_default,NLL,–,0.808 ± 0.009
1,credit_default,PO_Entropy,1.000,0.775 ± 0.027
2,credit_default,PO_Entropy,29.023,0.821 ± 0.002


In [18]:
df_tests_all

,dataset,method,beta,rep,test_accuracy,test_tpr,test_fnr,test_tnr,test_bal_acc,test_auc,is_beta_star
0,credit_default,NLL,NaN,0,0.813167,0.272042,0.727958,0.966831,0.619436,0.696444,False
1,credit_default,NLL,NaN,1,0.813500,0.276564,0.723436,0.965975,0.621269,0.709061,False
2,credit_default,NLL,NaN,2,0.797167,0.151469,0.848531,0.980526,0.565998,0.692989,False
3,credit_default,PO_Entropy,1.000000,0,0.743833,0.082894,0.917106,0.931522,0.507208,0.523679,False
4,credit_default,PO_Entropy,1.000000,1,0.788333,0.113791,0.886209,0.979884,0.546837,0.634651,False
5,credit_default,PO_Entropy,1.000000,2,0.793167,0.278824,0.721176,0.939225,0.609025,0.679495,False
6,credit_default,PO_Entropy,29.022795,0,0.819167,0.359457,0.640543,0.949711,0.654584,0.710821,True
7,credit_default,PO_Entropy,29.022795,1,0.822667,0.373022,0.626978,0.950353,0.661687,0.708354,True
8,credit_default,PO_Entropy,29.022795,2,0.821000,0.366240,0.633760,0.950139,0.658189,0.707585,True
